# 02 · Cleaning, Validation & Feature Derivation

> **Stage 2 (STRUCTURE.md).** Three jobs: **(A)** reproduce the documented raw→clean fixes and show the
> defects resolve to zero; **(B)** validate the *provided* clean file against business rules; **(C)** derive
> the analysis features from assessment §7 — while deliberately **not** deriving anything the data cannot support.

**Changelog vs. plan:** exact salary *values* cannot be reproduced row-for-row (the original median-fill seed
is unknown), so Part A validates the cleaning **methodology** — defect counts, monotonicity, floors — not
literal equality. Part B validates the clean file we were actually given.

In [1]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath("../src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import mck_style
mck_style.apply()
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

ARCH = "../archive"; PROC = "../data/processed"; FIG = "../reports/figures"
os.makedirs(PROC, exist_ok=True); os.makedirs(FIG, exist_ok=True)
np.random.seed(42)
print("Setup complete · pandas", pd.__version__)

Setup complete · pandas 3.0.3


## Part A — Reproduce the raw→clean methodology

The raw file carries three documented defect classes (assessment §2–3, archive readme). We fix each with the
technique the dataset author states they used, then confirm the defect count drops to zero.

In [2]:
raw = pd.read_csv(f"{ARCH}/hr_raw.csv",
    dtype={"Department":"category","Job_Level":"category","Job_Title":"category",
           "Status":"category","Work_Mode":"category","Performance_Rating":"object",
           "Country":"category","City":"category"},
    parse_dates=["Hire_Date"])
before = {
    "Perf_Rating nulls":       int(raw["Performance_Rating"].isnull().sum()),
    "Salary <= 0":             int((raw["Salary"] <= 0).sum()),
    "Age-Exp gap < 22":        int(((raw["Age"] - raw["Experience_Years"]) < 22).sum()),
}
pd.Series(before).to_frame("defects BEFORE")

,defects BEFORE
Perf_Rating nulls,3333
Salary <= 0,3333
Age-Exp gap < 22,164794


**Fix 1 — Multi-level median fill for invalid salaries.** Non-positive salaries → NaN → filled with the median of their `Job_Level × Department` group (the author's stated rule).

In [3]:
raw["Salary"] = raw["Salary"].mask(raw["Salary"] <= 0)
grp_median = raw.groupby(["Job_Level","Department"])["Salary"].transform("median")
raw["Salary"] = raw["Salary"].fillna(grp_median)
print("Salary <= 0 after fix:", int((raw["Salary"] <= 0).sum()))

Salary <= 0 after fix: 0


**Fix 2 — Stratified-sample fill for missing performance ratings.** Each null draws from the observed rating distribution *within its Department*, preserving departmental proportions (the author's stated rule).

In [4]:
def stratified_fill(s):
    obs = s.dropna()
    n = s.isnull().sum()
    if n == 0:
        return s
    fill = np.random.choice(obs.values, size=n, replace=True)
    s = s.copy(); s.loc[s.isnull()] = fill
    return s
raw["Performance_Rating"] = (raw.groupby("Department", group_keys=False)["Performance_Rating"]
                                .apply(stratified_fill))
print("Perf_Rating nulls after fix:", int(raw["Performance_Rating"].isnull().sum()))

Perf_Rating nulls after fix: 0


**Fix 3 — Age–experience alignment.** Where `Age < Experience + 22`, lift `Age` to the `Experience + 22` floor (the author's stated baseline). Note: the *provided* clean file only enforces a gap of **21**, a documented rule-vs-reality discrepancy we surface in Part B.

In [5]:
viol = (raw["Age"] - raw["Experience_Years"]) < 22
raw.loc[viol, "Age"] = raw.loc[viol, "Experience_Years"] + 22
print("Age-Exp gap < 22 after fix:", int(((raw["Age"] - raw["Experience_Years"]) < 22).sum()))
print("New minimum gap:", int((raw["Age"] - raw["Experience_Years"]).min()))

Age-Exp gap < 22 after fix: 0
New minimum gap: 22


**Fix 4 — Derive `Hire_Year` and cast types** (raw lacks `Hire_Year`; the clean schema adds it).

In [6]:
raw["Hire_Year"] = raw["Hire_Date"].dt.year.astype("int16")
raw["Experience_Years"] = raw["Experience_Years"].astype("int16")
after = {
    "Perf_Rating nulls":  int(raw["Performance_Rating"].isnull().sum()),
    "Salary <= 0":        int((raw["Salary"] <= 0).sum()),
    "Age-Exp gap < 22":   int(((raw["Age"] - raw["Experience_Years"]) < 22).sum()),
}
pd.DataFrame({"BEFORE": before, "AFTER": after})

,BEFORE,AFTER
Perf_Rating nulls,3333,0
Salary <= 0,3333,0
Age-Exp gap < 22,164794,0


> **So What:** all three documented defect classes resolve to **0** with the author's stated techniques,
> and post-fix median salary is monotonic by level — the cleaning recipe is reproducible and auditable.
> We now discard our reconstruction and validate the *official* clean file, the single source of truth downstream.

In [7]:
mono = raw.pivot_table("Salary","Job_Level","Department","median").round(0)
mono = mono.reindex(["Junior","Mid","Senior","Director"])
print("Reproduced median salary monotonic (Junior<Mid<Senior<Director) per dept:",
      bool((mono.diff().dropna() > 0).all().all()))
del raw

Reproduced median salary monotonic (Junior<Mid<Senior<Director) per dept: True


## Part B — Validate the *provided* clean file

Load the Parquet from Stage 1 and assert every business rule.

In [8]:
clean = pd.read_parquet(f"{PROC}/hr_clean.parquet")
checks = {}
checks["no nulls"]            = int(clean.isnull().sum().sum()) == 0
checks["salary > 0"]          = bool((clean["Salary"] > 0).all())
checks["age in [21,66]"]      = bool(clean["Age"].between(21,66).all())
checks["experience in [0,30]"]= bool(clean["Experience_Years"].between(0,30).all())
med = clean.pivot_table("Salary","Job_Level","Department","median").reindex(
      ["Junior","Mid","Senior","Director"])
checks["salary monotonic by level"] = bool((med.diff().dropna() > 0).all().all())
pd.Series(checks).to_frame("PASS")

,PASS
no nulls,True
salary > 0,True
"age in [21,66]",True
"experience in [0,30]",True
salary monotonic by level,True


**Two documented data-reality flags** (assessment §3) — real limitations we carry into every downstream decision:

In [9]:
flags = {
    "Year == Hire_Year for every row": bool((clean["Year"] == clean["Hire_Year"]).all()),
    "Min Age-Exp gap (rule says 22)":  int((clean["Age"] - clean["Experience_Years"]).min()),
    "Gender column present":           "Gender" in clean.columns,
}
pd.Series(flags, dtype="object").to_frame("reality check")

,reality check
Year == Hire_Year for every row,True
Min Age-Exp gap (rule says 22),21
Gender column present,False


> **Implication:** `Year == Hire_Year` means there is **no snapshot date**, so *current* tenure and
> time-to-attrition are uncomputable; and there is **no `Gender`**, so pay-equity/DEI analysis is out of scope.
> These are hard boundaries, not modelling choices.

## Part C — Derive analysis features (assessment §7)

Eight defensible features. We **deliberately skip `Tenure_Years`** — it requires a snapshot date the data
does not contain (deriving it would fabricate a meaningless number).

In [10]:
# 1. Attrition flag (Resigned/Terminated = left the company)
clean["Attrition_Flag"] = clean["Status"].isin(["Resigned","Terminated"]).astype("int8")

# 2. Pay position vs Level x Dept median (pay-equity / outlier ratio)
lvl_dept_med = clean.groupby(["Job_Level","Department"])["Salary"].transform("median")
clean["Salary_vs_Level_Dept_Median"] = (clean["Salary"] / lvl_dept_med).astype("float32")

# 3. Salary per year of experience (guard divide-by-zero)
exp = clean["Experience_Years"].replace(0, np.nan)
clean["Salary_per_Experience_Year"] = (clean["Salary"] / exp).astype("float32")

# 4. Age band & generation cohort
clean["Age_Band"] = pd.cut(clean["Age"], [20,29,39,49,66],
                           labels=["21-29","30-39","40-49","50-66"])
clean["Generation_Cohort"] = pd.cut(clean["Age"], [20,27,43,66],
                           labels=["Gen Z","Millennial","Gen X+"])

# 5. First / last name split
parts = clean["Full_Name"].str.split(n=1, expand=True)
clean["First_Name"] = parts[0]; clean["Last_Name"] = parts[1]

# 6. Region grouping (Western vs Southern vs Central/Eastern Europe)
region_map = {"United Kingdom":"Western","France":"Western","Netherlands":"Western",
              "Germany":"Central","Poland":"Central","Spain":"Southern","Italy":"Southern"}
clean["Region"] = clean["Country"].map(region_map).astype("category")

# 7-8. Hire decade & quarter (cohort / seasonality)
clean["Hire_Decade"] = (clean["Hire_Year"]//10*10).astype("int16")
clean["Hire_Quarter"] = ("Q" + clean["Hire_Date"].dt.quarter.astype("str")).astype("category")

new_cols = ["Attrition_Flag","Salary_vs_Level_Dept_Median","Salary_per_Experience_Year",
            "Age_Band","Generation_Cohort","First_Name","Last_Name","Region",
            "Hire_Decade","Hire_Quarter"]
clean[new_cols].head()

,Attrition_Flag,Salary_vs_Level_Dept_Median,Salary_per_Experience_Year,Age_Band,Generation_Cohort,First_Name,Last_Name,Region,Hire_Decade,Hire_Quarter
0,0,1.06,"11,624.00",30-39,Millennial,Heinz-Georg,Eimer,Central,2020,Q1
1,0,0.96,"10,210.73",40-49,Millennial,Maartje,van den Nuwenhuysen-Geertsen,Western,2000,Q4
2,0,0.95,"7,408.07",40-49,Millennial,Sara,Sureda Figueroa,Southern,2010,Q1
3,0,1.04,"49,012.00",21-29,Gen Z,Luce,Sanchez,Western,2020,Q2
4,0,1.14,"28,776.50",30-39,Millennial,William,Jennings,Western,2020,Q4


In [11]:
print("Blocked feature — Tenure_Years — intentionally NOT derived (no snapshot date).")
clean.to_parquet(f"{PROC}/hr_features.parquet", index=False)
print("Wrote", f"{PROC}/hr_features.parquet", "· shape", clean.shape)

Blocked feature — Tenure_Years — intentionally NOT derived (no snapshot date).


Wrote ../data/processed/hr_features.parquet · shape (2000000, 26)


### Stage 2 — Gate checklist ✅
- [x] All three raw defect classes reproduced and resolved to 0 (methodology auditable)
- [x] Provided clean file passes every business rule (nulls, ranges, salary monotonicity)
- [x] Data-reality limits surfaced: no snapshot date, no `Gender` (rule-vs-reality gap of 22→21 noted)
- [x] 8 analysis features derived; `Tenure_Years` correctly **not** derived
- [x] `data/processed/hr_features.parquet` written

→ Proceed to **03 · EDA**.